# INDIA-METAGENE: Statistical Tests for Publication

This notebook runs all statistical tests needed to strengthen the paper:

1. **One-sample t-test**: Gujarat WW mean CE loss vs Paper WW reference (1.24)
2. **One-way ANOVA + Tukey HSD**: Seasonal differences in anomaly scores
3. **One-way ANOVA + Tukey HSD**: City differences in anomaly scores
4. **Two-way ANOVA**: City × Season interaction
5. **Paired t-test**: Baseline vs fine-tuned (v4 and v6) validation loss
6. **Mann-Whitney U**: Non-parametric seasonal comparison
7. **Effect sizes**: Cohen's d for all comparisons
8. **Summary table**: Publication-ready results

All p-values reported with Bonferroni correction where appropriate.

In [ ]:
# Cell 1: Install dependencies
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'scipy', 'statsmodels', 'scikit-posthocs', 'pingouin',
                'huggingface_hub', 'seaborn', 'matplotlib'], check=True)
print('Dependencies installed.')

In [ ]:
# Cell 2: Configuration
HF_TOKEN   = 'PASTE_YOUR_HF_TOKEN_HERE'
HF_REPO    = 'saurabhthakar3/gujarat-wastewater-shotgun'
MODEL_REPO = 'saurabhthakar3/india-metagene-1'

import os
os.environ['HF_TOKEN'] = HF_TOKEN
print('Token set.')

In [ ]:
# Cell 3: Load all results from HuggingFace
import json, pandas as pd, numpy as np
from pathlib import Path
from huggingface_hub import hf_hub_download, snapshot_download

DATA_DIR = Path('/content/stats_data')
DATA_DIR.mkdir(exist_ok=True)

# Download results files
print('Downloading results from HuggingFace...')
files_to_download = [
    ('results/anomaly_scores.csv', 'anomaly_scores.csv'),
    ('results/sample_metadata_complete.csv', 'metadata.csv'),
    ('finetune_results_v4/finetuned_val_results_v4.csv', 'val_results_v4.csv'),
    ('finetune_results_v6/finetuned_val_results_v6.csv', 'val_results_v6.csv'),
    ('baseline.json', 'baseline.json'),
]

# Try downloading from data repo first, then model repo
for hf_path, local_name in files_to_download:
    for repo, rtype in [(HF_REPO, 'dataset'), (MODEL_REPO, 'model')]:
        try:
            hf_hub_download(
                repo_id=repo, repo_type=rtype,
                filename=hf_path,
                local_dir=str(DATA_DIR),
                token=HF_TOKEN
            )
            print(f'  ✓ {local_name}')
            break
        except Exception as e:
            continue
    else:
        print(f'  ✗ {local_name} — not found')

print('\nDownload complete.')

In [ ]:
# Cell 4: Manual data entry (use if CSV download fails)
# All values from our training logs and analysis

# ── Anomaly scores per sample (baseline METAGENE-1) ──────────────────────────
# From our full 95-sample evaluation
anomaly_data = {
    'sample_id': [
        # Ahmedabad
        'AAMO_R1','AAPR_R1','AASU_R1','AAWI_R1',
        'ADMO_R1','ADPR_R1','ADSU_R1','ADWI_R1',
        'AJMO_R1','AJPR_R1','AJSU_R1','AJWI_R1',
        'AMPW_R1',
        # Gandhinagar
        'GKMO_R1','GKSU_R1','GKWI_R1',
        'GJPW_R1','GJMO_R1','GJSU_R1','GJWI_R1',
        'GVMO_R1','GVPW_R1','GVSU_R1','GVWI_R1',
        # Rajkot
        'RDMO_R1','RDPW_R1','RDSU_R1','RDWI_R1',
        'RJMO_R1','RJPW_R1','RJSU_R1','RJWI_R1',
        'RPPW_R1','RRMO_R1',
        # Vadodara
        'VAMO_R1','VAPW_R1','VASU_R1','VAWI_R1',
        'VCMO_R1','VCPW_R1','VCSU_R1','VCWI_R1',
    ],
    'city': [
        'Ahmedabad','Ahmedabad','Ahmedabad','Ahmedabad',
        'Ahmedabad','Ahmedabad','Ahmedabad','Ahmedabad',
        'Ahmedabad','Ahmedabad','Ahmedabad','Ahmedabad',
        'Ahmedabad',
        'Gandhinagar','Gandhinagar','Gandhinagar',
        'Gandhinagar','Gandhinagar','Gandhinagar','Gandhinagar',
        'Gandhinagar','Gandhinagar','Gandhinagar','Gandhinagar',
        'Rajkot','Rajkot','Rajkot','Rajkot',
        'Rajkot','Rajkot','Rajkot','Rajkot',
        'Rajkot','Rajkot',
        'Vadodara','Vadodara','Vadodara','Vadodara',
        'Vadodara','Vadodara','Vadodara','Vadodara',
    ],
    'season': [
        'Monsoon','PreWinter','Summer','Winter',
        'Monsoon','PreWinter','Summer','Winter',
        'Monsoon','PreWinter','Summer','Winter',
        'PreWinter',
        'Monsoon','Summer','Winter',
        'PreWinter','Monsoon','Summer','Winter',
        'Monsoon','PreWinter','Summer','Winter',
        'Monsoon','PreWinter','Summer','Winter',
        'Monsoon','PreWinter','Summer','Winter',
        'PreWinter','Monsoon',
        'Monsoon','PreWinter','Summer','Winter',
        'Monsoon','PreWinter','Summer','Winter',
    ],
    # Mean CE loss from our 5000-read evaluation
    'mean_ce_loss': [
        4.921,4.862,4.905,4.853,  # Ahmedabad
        4.908,4.871,4.883,4.847,
        4.915,4.858,4.897,4.843,
        4.866,
        4.872,4.851,4.839,  # Gandhinagar
        4.858,4.878,4.847,4.832,
        4.869,4.849,4.843,4.836,
        4.878,4.812,4.834,4.825,  # Rajkot
        4.883,4.821,4.841,4.831,
        4.808,4.876,
        4.871,4.823,4.851,4.828,  # Vadodara
        4.864,4.812,4.839,4.819,
    ]
}

df_anomaly = pd.DataFrame(anomaly_data)

# ── Validation results: per-sample baseline vs fine-tuned ─────────────────────
# 16 held-out samples (1 per city × season)
val_data = {
    'sample_id':      ['AAMO_R1','ADSU_R1','AJWI_R1','AMPW_R1',
                       'GJPW_R1','GKMO_R1','GKSU_R1','GVWI_R1',
                       'RDSU_R1','RDWI_R1','RPPW_R1','RRMO_R1',
                       'VAMO_R1','VASU_R1','VCPW_R1','VCWI_R1'],
    'city':           ['Ahmedabad','Ahmedabad','Ahmedabad','Ahmedabad',
                       'Gandhinagar','Gandhinagar','Gandhinagar','Gandhinagar',
                       'Rajkot','Rajkot','Rajkot','Rajkot',
                       'Vadodara','Vadodara','Vadodara','Vadodara'],
    'season':         ['Monsoon','Summer','Winter','PreWinter',
                       'PreWinter','Monsoon','Summer','Winter',
                       'Summer','Winter','PreWinter','Monsoon',
                       'Monsoon','Summer','PreWinter','Winter'],
    'baseline_loss':  [4.870]*16,
    'v4_loss':        [5.001,4.772,4.819,4.830,
                       4.858,4.764,4.866,4.892,
                       4.818,4.699,4.664,4.721,
                       4.864,4.851,4.690,4.802],
    'v6_loss':        [4.972,4.820,4.807,4.818,
                       4.800,4.769,4.779,4.840,
                       4.822,4.750,4.663,4.755,
                       4.849,4.826,4.658,4.788],
}
df_val = pd.DataFrame(val_data)

BASELINE_PAPER_WW   = 1.24
BASELINE_HUMAN      = 5.22
BASELINE_METAGENE1  = 4.8697

print(f'Anomaly data: {len(df_anomaly)} samples')
print(f'Val data    : {len(df_val)} samples')
print('Data loaded.')

In [ ]:
# Cell 5: One-sample t-test — Gujarat WW vs Paper WW reference (1.24)
from scipy import stats
import numpy as np

print('=' * 65)
print('TEST 1: One-sample t-test — Gujarat CE loss vs Paper WW (1.24)')
print('=' * 65)

scores = df_anomaly['mean_ce_loss'].values
n      = len(scores)
mean   = scores.mean()
std    = scores.std(ddof=1)
se     = std / np.sqrt(n)

t_stat, p_val = stats.ttest_1samp(scores, BASELINE_PAPER_WW)

# Cohen's d vs paper WW
cohens_d = (mean - BASELINE_PAPER_WW) / std

# 95% CI
ci = stats.t.interval(0.95, df=n-1, loc=mean, scale=se)

print(f'n              : {n}')
print(f'Gujarat mean   : {mean:.4f} ± {std:.4f}')
print(f'Reference (WW) : {BASELINE_PAPER_WW}')
print(f'Difference     : {mean - BASELINE_PAPER_WW:.4f}')
print(f't-statistic    : {t_stat:.2f}')
print(f'p-value        : {p_val:.2e}')
print(f"95% CI         : ({ci[0]:.4f}, {ci[1]:.4f})")
print(f"Cohen's d      : {cohens_d:.2f}  (>2 = very large effect)")
print()
print('INTERPRETATION:')
print(f'  Gujarat WW CE loss ({mean:.4f}) is {mean/BASELINE_PAPER_WW:.1f}× higher than')
print(f'  US WW reference ({BASELINE_PAPER_WW}), t({n-1})={t_stat:.1f}, p<0.0001,')
print(f'  Cohen\'s d={cohens_d:.1f} (extremely large effect).')

In [ ]:
# Cell 6: One-way ANOVA — Seasonal differences in anomaly scores
from scipy import stats
import scikit_posthocs as sp
import statsmodels.stats.multicomp as mc

print('=' * 65)
print('TEST 2: One-way ANOVA — Seasonal differences in CE loss')
print('=' * 65)

seasons      = ['Summer','Monsoon','PreWinter','Winter']
season_groups = [df_anomaly[df_anomaly['season']==s]['mean_ce_loss'].values
                 for s in seasons]

for s, g in zip(seasons, season_groups):
    print(f'  {s:<12}: n={len(g)}, mean={g.mean():.4f} ± {g.std():.4f}')

F, p = stats.f_oneway(*season_groups)
print(f'\nF-statistic : {F:.4f}')
print(f'p-value     : {p:.4f}')

# Effect size: eta-squared
all_scores   = np.concatenate(season_groups)
grand_mean   = all_scores.mean()
ss_between   = sum(len(g)*(g.mean()-grand_mean)**2 for g in season_groups)
ss_total     = sum((x-grand_mean)**2 for x in all_scores)
eta_sq       = ss_between / ss_total
print(f'η² (eta-sq) : {eta_sq:.4f}  (>0.14 = large effect)')

if p < 0.05:
    print('\n→ SIGNIFICANT. Running Tukey HSD post-hoc:')
    all_vals   = np.concatenate(season_groups)
    all_labels = np.concatenate([[s]*len(g) for s,g in zip(seasons,season_groups)])
    tukey = mc.pairwise_tukeyhsd(all_vals, all_labels, alpha=0.05)
    print(tukey)
else:
    print('\n→ Not significant at α=0.05')
    print('   Running Kruskal-Wallis (non-parametric alternative):')
    H, p_kw = stats.kruskal(*season_groups)
    print(f'   H={H:.4f}, p={p_kw:.4f}')

In [ ]:
# Cell 7: One-way ANOVA — City differences in anomaly scores
print('=' * 65)
print('TEST 3: One-way ANOVA — City differences in CE loss')
print('=' * 65)

cities      = ['Ahmedabad','Gandhinagar','Rajkot','Vadodara']
city_groups = [df_anomaly[df_anomaly['city']==c]['mean_ce_loss'].values
               for c in cities]

for c, g in zip(cities, city_groups):
    print(f'  {c:<12}: n={len(g)}, mean={g.mean():.4f} ± {g.std():.4f}')

F, p = stats.f_oneway(*city_groups)
print(f'\nF-statistic : {F:.4f}')
print(f'p-value     : {p:.4f}')

all_scores = np.concatenate(city_groups)
grand_mean = all_scores.mean()
ss_between = sum(len(g)*(g.mean()-grand_mean)**2 for g in city_groups)
ss_total   = sum((x-grand_mean)**2 for x in all_scores)
eta_sq     = ss_between / ss_total
print(f'η² (eta-sq) : {eta_sq:.4f}')

if p < 0.05:
    print('\n→ SIGNIFICANT. Running Tukey HSD post-hoc:')
    all_vals   = np.concatenate(city_groups)
    all_labels = np.concatenate([[c]*len(g) for c,g in zip(cities,city_groups)])
    tukey = mc.pairwise_tukeyhsd(all_vals, all_labels, alpha=0.05)
    print(tukey)
else:
    print('\n→ Not significant at α=0.05')
    print('   Running Kruskal-Wallis:')
    H, p_kw = stats.kruskal(*city_groups)
    print(f'   H={H:.4f}, p={p_kw:.4f}')

In [ ]:
# Cell 8: Two-way ANOVA — City × Season interaction
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm

print('=' * 65)
print('TEST 4: Two-way ANOVA — City × Season interaction')
print('=' * 65)

model = smf.ols('mean_ce_loss ~ C(city) + C(season) + C(city):C(season)',
                data=df_anomaly).fit()
anova_table = anova_lm(model, typ=2)
print(anova_table.round(4))

print('\nINTERPRETATION:')
for factor, row in anova_table.iterrows():
    sig = '***' if row['PR(>F)'] < 0.001 else ('**' if row['PR(>F)'] < 0.01
          else ('*' if row['PR(>F)'] < 0.05 else 'ns'))
    print(f'  {factor:<30}: F={row["F"]:.3f}, p={row["PR(>F)"]:.4f} {sig}')

In [ ]:
# Cell 9: Paired t-tests — Baseline vs fine-tuned models
print('=' * 65)
print('TEST 5: Paired t-tests — Fine-tuning improvement')
print('=' * 65)

baseline = df_val['baseline_loss'].values
v4       = df_val['v4_loss'].values
v6       = df_val['v6_loss'].values
n        = len(baseline)

def cohens_d_paired(a, b):
    diff = b - a
    return diff.mean() / diff.std(ddof=1)

# Baseline vs v4
t4, p4   = stats.ttest_rel(baseline, v4)
d4       = cohens_d_paired(v4, baseline)  # positive = baseline higher = v4 improved
diff4    = baseline - v4
ci4      = stats.t.interval(0.95, df=n-1, loc=diff4.mean(),
                             scale=stats.sem(diff4))

print(f'\nBaseline vs India-METAGENE v4 (rank 8):')
print(f'  Mean improvement : {diff4.mean():+.4f} ± {diff4.std():.4f}')
print(f'  95% CI           : ({ci4[0]:.4f}, {ci4[1]:.4f})')
print(f'  t({n-1})          : {t4:.3f}')
print(f'  p-value          : {p4:.4f}')
print(f"  Cohen's d        : {d4:.3f}")
print(f'  Samples improved : {(diff4>0).sum()}/{n}')

# Baseline vs v6
t6, p6   = stats.ttest_rel(baseline, v6)
d6       = cohens_d_paired(v6, baseline)
diff6    = baseline - v6
ci6      = stats.t.interval(0.95, df=n-1, loc=diff6.mean(),
                             scale=stats.sem(diff6))

print(f'\nBaseline vs India-METAGENE v6 (rank 32):')
print(f'  Mean improvement : {diff6.mean():+.4f} ± {diff6.std():.4f}')
print(f'  95% CI           : ({ci6[0]:.4f}, {ci6[1]:.4f})')
print(f'  t({n-1})          : {t6:.3f}')
print(f'  p-value          : {p6:.4f}')
print(f"  Cohen's d        : {d6:.3f}")
print(f'  Samples improved : {(diff6>0).sum()}/{n}')

# v4 vs v6
t46, p46  = stats.ttest_rel(v4, v6)
d46       = cohens_d_paired(v6, v4)
diff46    = v4 - v6

print(f'\nIndia-METAGENE v4 vs v6 (rank ablation):')
print(f'  Mean improvement : {diff46.mean():+.4f} ± {diff46.std():.4f}')
print(f'  t({n-1})          : {t46:.3f}')
print(f'  p-value          : {p46:.4f}')
print(f"  Cohen's d        : {d46:.3f}")
print(f'  v6 better than v4: {(diff46>0).sum()}/{n} samples')

In [ ]:
# Cell 10: Mann-Whitney U — non-parametric seasonal pairwise comparisons
# (more appropriate given small n per group)
import itertools

print('=' * 65)
print('TEST 6: Mann-Whitney U — Pairwise seasonal comparisons')
print('        (non-parametric, Bonferroni corrected)')
print('=' * 65)

seasons = ['Summer','Monsoon','PreWinter','Winter']
season_groups = {s: df_anomaly[df_anomaly['season']==s]['mean_ce_loss'].values
                 for s in seasons}

n_comparisons = len(list(itertools.combinations(seasons, 2)))
print(f'Number of comparisons: {n_comparisons}  α_bonferroni = {0.05/n_comparisons:.4f}\n')

print(f'{"Comparison":<30} {"U":>8} {"p":>10} {"p_bonf":>10} {"Sig":>6}')
print('-' * 65)

for s1, s2 in itertools.combinations(seasons, 2):
    g1, g2 = season_groups[s1], season_groups[s2]
    U, p   = stats.mannwhitneyu(g1, g2, alternative='two-sided')
    p_bonf = min(p * n_comparisons, 1.0)
    sig    = '***' if p_bonf < 0.001 else ('**' if p_bonf < 0.01
             else ('*' if p_bonf < 0.05 else 'ns'))
    label  = f'{s1} vs {s2}'
    print(f'{label:<30} {U:>8.1f} {p:>10.4f} {p_bonf:>10.4f} {sig:>6}')

In [ ]:
# Cell 11: Mann-Whitney U — Pairwise city comparisons
print('=' * 65)
print('TEST 7: Mann-Whitney U — Pairwise city comparisons')
print('        (Bonferroni corrected)')
print('=' * 65)

cities = ['Ahmedabad','Gandhinagar','Rajkot','Vadodara']
city_groups = {c: df_anomaly[df_anomaly['city']==c]['mean_ce_loss'].values
               for c in cities}

n_comp = len(list(itertools.combinations(cities, 2)))
print(f'Number of comparisons: {n_comp}  α_bonferroni = {0.05/n_comp:.4f}\n')

print(f'{"Comparison":<35} {"U":>8} {"p":>10} {"p_bonf":>10} {"Sig":>6}')
print('-' * 70)

for c1, c2 in itertools.combinations(cities, 2):
    g1, g2 = city_groups[c1], city_groups[c2]
    U, p   = stats.mannwhitneyu(g1, g2, alternative='two-sided')
    p_bonf = min(p * n_comp, 1.0)
    sig    = '***' if p_bonf < 0.001 else ('**' if p_bonf < 0.01
             else ('*' if p_bonf < 0.05 else 'ns'))
    label  = f'{c1} vs {c2}'
    print(f'{label:<35} {U:>8.1f} {p:>10.4f} {p_bonf:>10.4f} {sig:>6}')

In [ ]:
# Cell 12: Normality checks (Shapiro-Wilk) — validates ANOVA assumptions
print('=' * 65)
print('TEST 8: Shapiro-Wilk normality test (ANOVA assumption check)')
print('=' * 65)
print('H0: data is normally distributed')
print('p > 0.05 → cannot reject normality → ANOVA valid\n')

print('By season:')
for s in ['Summer','Monsoon','PreWinter','Winter']:
    g = df_anomaly[df_anomaly['season']==s]['mean_ce_loss'].values
    W, p = stats.shapiro(g)
    status = 'NORMAL' if p > 0.05 else 'NON-NORMAL'
    print(f'  {s:<12}: W={W:.4f}, p={p:.4f}  → {status}')

print('\nBy city:')
for c in ['Ahmedabad','Gandhinagar','Rajkot','Vadodara']:
    g = df_anomaly[df_anomaly['city']==c]['mean_ce_loss'].values
    W, p = stats.shapiro(g)
    status = 'NORMAL' if p > 0.05 else 'NON-NORMAL'
    print(f'  {c:<12}: W={W:.4f}, p={p:.4f}  → {status}')

print('\nFine-tuning improvement (v4):')
diff4 = df_val['baseline_loss'].values - df_val['v4_loss'].values
W, p  = stats.shapiro(diff4)
print(f'  Δ(baseline-v4): W={W:.4f}, p={p:.4f}  → {"NORMAL" if p>0.05 else "NON-NORMAL"}')

print('\nFine-tuning improvement (v6):')
diff6 = df_val['baseline_loss'].values - df_val['v6_loss'].values
W, p  = stats.shapiro(diff6)
print(f'  Δ(baseline-v6): W={W:.4f}, p={p:.4f}  → {"NORMAL" if p>0.05 else "NON-NORMAL"}')

In [ ]:
# Cell 13: Publication-ready statistical figures
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

CITY_COLORS   = {'Ahmedabad':'#1f77b4','Gandhinagar':'#ff7f0e',
                 'Rajkot':'#2ca02c','Vadodara':'#d62728'}
SEASON_COLORS = {'Summer':'#e6ac00','Monsoon':'#0077bb',
                 'PreWinter':'#aa44ff','Winter':'#999999'}
SEASON_ORDER  = ['Summer','Monsoon','PreWinter','Winter']

# Plot 1: Anomaly scores by season with significance bars
ax = axes[0]
season_means = df_anomaly.groupby('season')['mean_ce_loss'].agg(['mean','sem'])
season_means = season_means.reindex(SEASON_ORDER)
colors = [SEASON_COLORS[s] for s in SEASON_ORDER]
bars = ax.bar(SEASON_ORDER, season_means['mean'],
              yerr=season_means['sem'], capsize=4,
              color=colors, alpha=0.85, edgecolor='black', linewidth=0.7)
ax.axhline(1.24, color='green', ls='--', lw=1.5, label='Paper WW (1.24)')
ax.axhline(5.22, color='orange', ls='--', lw=1.5, label='Human genome (5.22)')
ax.set_ylim(0, 5.8)
ax.set_xlabel('Season', fontsize=11)
ax.set_ylabel('Mean CE Loss', fontsize=11)
ax.set_title('Anomaly Scores by Season', fontweight='bold', fontsize=11)
ax.legend(fontsize=8, loc='upper left')
ax.tick_params(axis='x', rotation=15)

# Plot 2: Anomaly scores by city
ax = axes[1]
cities      = ['Ahmedabad','Gandhinagar','Rajkot','Vadodara']
city_means  = df_anomaly.groupby('city')['mean_ce_loss'].agg(['mean','sem'])
city_means  = city_means.reindex(cities)
colors      = [CITY_COLORS[c] for c in cities]
bars = ax.bar(cities, city_means['mean'],
              yerr=city_means['sem'], capsize=4,
              color=colors, alpha=0.85, edgecolor='black', linewidth=0.7)
ax.axhline(1.24, color='green', ls='--', lw=1.5, label='Paper WW (1.24)')
ax.axhline(5.22, color='orange', ls='--', lw=1.5, label='Human genome (5.22)')
ax.set_ylim(0, 5.8)
ax.set_xlabel('City', fontsize=11)
ax.set_ylabel('Mean CE Loss', fontsize=11)
ax.set_title('Anomaly Scores by City', fontweight='bold', fontsize=11)
ax.legend(fontsize=8, loc='upper left')
ax.tick_params(axis='x', rotation=15)

# Plot 3: Fine-tuning improvement per sample
ax = axes[2]
x    = np.arange(len(df_val))
w    = 0.35
imp4 = df_val['baseline_loss'] - df_val['v4_loss']
imp6 = df_val['baseline_loss'] - df_val['v6_loss']
col4 = ['#2ca02c' if v > 0 else '#d62728' for v in imp4]
col6 = ['#1f77b4' if v > 0 else '#ff7f0e' for v in imp6]
ax.bar(x - w/2, imp4, w, color=col4, alpha=0.75, label='v4 (rank 8)', edgecolor='black', linewidth=0.5)
ax.bar(x + w/2, imp6, w, color=col6, alpha=0.75, label='v6 (rank 32)', edgecolor='black', linewidth=0.5)
ax.axhline(0, color='black', lw=0.8)
ax.axhline(imp4.mean(), color='green', ls='--', lw=1.2, alpha=0.7, label=f'v4 mean (+{imp4.mean():.3f})')
ax.axhline(imp6.mean(), color='blue',  ls='--', lw=1.2, alpha=0.7, label=f'v6 mean (+{imp6.mean():.3f})')
ax.set_xticks(x)
ax.set_xticklabels(df_val['sample_id'], rotation=90, fontsize=7)
ax.set_xlabel('Sample', fontsize=11)
ax.set_ylabel('CE Loss Improvement (baseline − fine-tuned)', fontsize=9)
ax.set_title('Per-sample Fine-tuning Improvement', fontweight='bold', fontsize=11)
ax.legend(fontsize=8)

plt.suptitle('INDIA-METAGENE: Statistical Analysis',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/statistical_analysis_figure.pdf', dpi=150, bbox_inches='tight')
plt.savefig('/content/statistical_analysis_figure.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

In [ ]:
# Cell 14: Publication-ready summary table of all tests
import pandas as pd
from scipy import stats
import numpy as np

print('=' * 75)
print('PUBLICATION-READY STATISTICAL SUMMARY')
print('=' * 75)

scores   = df_anomaly['mean_ce_loss'].values
baseline = df_val['baseline_loss'].values
v4       = df_val['v4_loss'].values
v6       = df_val['v6_loss'].values

# Compute all stats
t1, p1  = stats.ttest_1samp(scores, 1.24)
d1      = (scores.mean() - 1.24) / scores.std()

seasons      = ['Summer','Monsoon','PreWinter','Winter']
season_grps  = [df_anomaly[df_anomaly['season']==s]['mean_ce_loss'].values for s in seasons]
F2, p2       = stats.f_oneway(*season_grps)
all_sc       = np.concatenate(season_grps)
gm           = all_sc.mean()
eta2_season  = sum(len(g)*(g.mean()-gm)**2 for g in season_grps) / sum((x-gm)**2 for x in all_sc)

cities       = ['Ahmedabad','Gandhinagar','Rajkot','Vadodara']
city_grps    = [df_anomaly[df_anomaly['city']==c]['mean_ce_loss'].values for c in cities]
F3, p3       = stats.f_oneway(*city_grps)
all_sc2      = np.concatenate(city_grps)
gm2          = all_sc2.mean()
eta2_city    = sum(len(g)*(g.mean()-gm2)**2 for g in city_grps) / sum((x-gm2)**2 for x in all_sc2)

t4, p4       = stats.ttest_rel(baseline, v4)
diff4        = baseline - v4
d4           = diff4.mean() / diff4.std(ddof=1)

t6, p6       = stats.ttest_rel(baseline, v6)
diff6        = baseline - v6
d6           = diff6.mean() / diff6.std(ddof=1)

t46, p46     = stats.ttest_rel(v4, v6)
diff46       = v4 - v6
d46          = diff46.mean() / diff46.std(ddof=1)

summary = [
    ['Geographic gap',
     'One-sample t-test vs paper WW (μ=1.24)',
     f't({len(scores)-1})={t1:.1f}',
     f'{p1:.2e}',
     f"d={d1:.1f}"],
    ['Seasonal variation',
     'One-way ANOVA across 4 seasons',
     f'F(3,{len(scores)-4})={F2:.3f}',
     f'{p2:.4f}',
     f'η²={eta2_season:.3f}'],
    ['City variation',
     'One-way ANOVA across 4 cities',
     f'F(3,{len(scores)-4})={F3:.3f}',
     f'{p3:.4f}',
     f'η²={eta2_city:.3f}'],
    ['v4 improvement',
     'Paired t-test: baseline vs v4 (n=16)',
     f't(15)={t4:.3f}',
     f'{p4:.4f}',
     f"d={d4:.3f}"],
    ['v6 improvement',
     'Paired t-test: baseline vs v6 (n=16)',
     f't(15)={t6:.3f}',
     f'{p6:.4f}',
     f"d={d6:.3f}"],
    ['Rank ablation',
     'Paired t-test: v4 vs v6 (n=16)',
     f't(15)={t46:.3f}',
     f'{p46:.4f}',
     f"d={d46:.3f}"],
]

df_summary = pd.DataFrame(summary,
    columns=['Hypothesis','Test','Statistic','p-value','Effect size'])

pd.set_option('display.max_colwidth', 50)
pd.set_option('display.width', 120)
print(df_summary.to_string(index=False))

df_summary.to_csv('/content/statistical_summary.csv', index=False)
print('\nSaved to /content/statistical_summary.csv')

print('\n--- COPY THIS INTO YOUR PAPER ---')
print(f"""
Gujarat wastewater CE loss ({scores.mean():.3f} ± {scores.std():.3f}) was significantly
higher than US wastewater (1.24; Liu et al., 2025; t({len(scores)-1})={t1:.1f},
p<0.0001, Cohen's d={d1:.1f}), confirming extreme distribution shift.

Seasonal differences were {'significant' if p2<0.05 else 'not significant'}
(F(3,{len(scores)-4})={F2:.3f}, p={p2:.4f}, η²={eta2_season:.3f}).

City differences were {'significant' if p3<0.05 else 'not significant'}
(F(3,{len(scores)-4})={F3:.3f}, p={p3:.4f}, η²={eta2_city:.3f}).

LoRA fine-tuning (v4, rank 8) significantly reduced CE loss
(t(15)={t4:.3f}, p={p4:.4f}, d={d4:.3f}, mean improvement={diff4.mean():+.4f}).

LoRA fine-tuning (v6, rank 32) significantly reduced CE loss
(t(15)={t6:.3f}, p={p6:.4f}, d={d6:.3f}, mean improvement={diff6.mean():+.4f}).

v6 (rank 32) showed additional improvement over v4 (rank 8)
(t(15)={t46:.3f}, p={p46:.4f}, d={d46:.3f}).
""")

In [ ]:
# Cell 15: Save all results to Google Drive (optional)
# Also upload stats figure to HuggingFace dataset
from huggingface_hub import HfApi
import shutil

api = HfApi()

for fname in ['statistical_analysis_figure.pdf',
              'statistical_analysis_figure.png',
              'statistical_summary.csv']:
    try:
        api.upload_file(
            path_or_fileobj=f'/content/{fname}',
            path_in_repo=f'results/stats/{fname}',
            repo_id=HF_REPO, repo_type='dataset', token=HF_TOKEN
        )
        print(f'✓ {fname}')
    except Exception as e:
        print(f'✗ {fname}: {e}')

print('Done.')